In [ ]:
import sys
# !{sys.executable} -m pip install matplotlib
# !{sys.executable} -m pip install mrcfile
# !{sys.executable} -m pip install numpy
# !{sys.executable} -m pip install pandas
# !{sys.executable} -m pip install Pillow
# !{sys.executable} -m pip install POT
# !{sys.executable} -m pip install scipy
# !{sys.executable} -m pip install seaborn
# !{sys.executable} -m pip install numba
# !{sys.executable} -m pip install Bio
# !{sys.executable} -m pip install tornado==4.5.3
!pip freeze

In [ ]:
# # %matplotlib notebook
# import matplotlib.pyplot as plt
# import math
# import numpy as np
# import ot
# from matplotlib import collections  as mc
# from mpl_toolkits.mplot3d import Axes3D
# import random
# import time
# import seaborn as sns
# import json

# import mrcfile
# import pandas as pd
# import trn, coords
# import importlib
# importlib.reload(trn)

# import torch
# from unbalancedgw.vanilla_ugw_solver import exp_ugw_sinkhorn
# from unbalancedgw.vanilla_ugw_solver import log_ugw_sinkhorn
# from unbalancedgw._vanilla_utils import ugw_cost
# from unbalancedgw.utils import generate_measure
# from unbalancedgw._vanilla_utils import l2_distortion

# from scipy.spatial.transform import Rotation

# from Bio import PDB
# from Bio.PDB.vectors import Vector, rotmat

# plt.rcParams["figure.figsize"] = (10,10)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ot

import json
import os

from jgw import jgw_wrapper
from cryo_utils import sample_pdb, sample, generate_pdb_no_mrc, mrc_volume
from alignment import build_coup, find_optimal_alignment

In [ ]:
# file_path_extensions = ['A', 'B', 'C', 'E', 'F', 'H', 'I', 'J', 'K', 'L']
# file_path_origin = 'Data/jgw/bio/1i3q/1i3q_%s.pdb'
# volumes = np.array([1349, 788, 557, 309, 273, 309, 273, 456])
# n_points = 5000
# num_skip = 10

file_path_extensions = ['A', 'B', 'CHJKL']
file_path_origin = '../../../jgw/bio/1i3q/1i3q_%s.pdb'
volumes = np.array([1349, 788, 557, 309, 273, 309, 273, 456])
n_points = 5000
num_skip = 10

In [ ]:
points_x = []
index = []
for i in range(len(file_path_extensions)):
    file_path = file_path_origin%(file_path_extensions[i],)
#     num = int(n_points / volumes.sum() * volumes[i])
#     x, y, z = sample(file_path, 0.1, n_points, random_seed=1)
#     print(file_path, num)
#     points_x.append([])
#     for j in range(num):
#         index.append(i)
#         points_x[-1].append([x[j],y[j],z[j]])
#     points_x[-1] = np.array(points_x[-1])
    points_x.append(sample_pdb(file_path, num_skip))
    print(points_x[-1].shape)
    for j in range(points_x[-1].shape[0]):
        index.append(i)
    
    fig = plt.figure(figsize=(12, 12))
    ax = fig.add_subplot(projection='3d')
    ax.scatter(points_x[-1][:,2], points_x[-1][:,0], points_x[-1][:,1])
    plt.show()

In [ ]:
points_y = []
for i in range(len(points_x)):
    for j in range(points_x[i].shape[0]):
        points_y.append([points_x[i][j][0], points_x[i][j][1], points_x[i][j][2]])
points_y = np.array(points_y)

print(points_y.shape)

fig = plt.figure(figsize=(12, 12))
ax = fig.add_subplot(projection='3d')
ax.scatter(points_y[:,2], points_y[:,0], points_y[:,1])
plt.show()

In [ ]:
mu, dif, costs = jgw_wrapper(points_x, points_y, epsilon=1e5, eta=1, verbose=False, max_iter=50, diff_thrsh=1e-3)

In [ ]:
# d_y, i_y = generate_d([points_y])
# mu_y = [1/d_y.shape[0]] * d_y.shape[0]
# mu_y = np.array(mu_y)

In [ ]:
# d_x, i_x = generate_d(points_x)
# mu_x = [1/d_x.shape[0]] * d_x.shape[0]
# mu_x = np.array(mu_x)

In [ ]:
# plt.imshow(d_x)
# plt.show()
# plt.imshow(d_y)
# plt.show()

In [ ]:
# eta = 1
# import time
# alpha = 1e5
# cur_mu = np.outer(mu_x, mu_y)
# print(cur_mu.sum())
# dif = []
# costs = []
# mus = []
# for i in range(50):
#     t = time.time()
#     c = gamma(cur_mu, d_x, i_x, d_y)
#     c_o = c
#     c -= c.min()
#     c = (c**eta) * (cur_mu**(1-eta))
    
# #     plt.imshow(c)
# #     plt.show()
# #     print(c)
    
    
#     k = 1/c.sum() *c.shape[0] * c.shape[1]
#     c = c*k
#     alpha1=alpha*k
#     mu = ot.sinkhorn(mu_x, mu_y, c, alpha1)
#     print('#####')
#     print(i)
#     print(k)
#     print(abs(mu - cur_mu).sum())
#     print(mu.sum())
#     dif.append(abs(mu - cur_mu).sum())
#     costs.append(cost(c_o, cur_mu))
#     plt.imshow(cur_mu)
#     plt.show()
# #     plt.imshow(c)
#     plt.show()
#     cur_mu = mu
# #     if dif[-1] < 1e-4:
# #         cur_mu = np.random.rand(mu.shape[0], mu.shape[1])
# #         cur_mu /= cur_mu.sum()
#     mus.append(cur_mu)
#     print(time.time() - t)

In [ ]:
fig = plt.figure(figsize=(4, 4))
plt.plot(dif)
plt.yscale('log')
plt.ylabel('error')
plt.xlabel('iteration')
plt.show()

In [ ]:
plt.imshow(mu)
plt.show()

In [ ]:
base_path = '../../../embuild-server/embuild/'
name = '5A5T'

# f = open(base_path + name + "chains/hyper.json", 'r')
# hyper = json.load(f)
# f.close()

file_path_y = base_path + "maps/" + name + "_map.mrc"

# chain_ids = []
# file_paths_x = []
# for c in hyper["chain_files"]:
#     chain_ids.append(c.split('.')[0].split('_')[-1])
#     file_paths_x.append(base_path + name + "chains/maps/" + name + "_chain_" + chain_ids[-1] + "_map.mrc")
chain_ids = ['A', 'C', 'E', 'F', 'H', 'K', 'L', 'M']
file_paths_x = [base_path + name + "chains/maps/" + name + "_chain_" + chain_id + "_map.mrc" for chain_id in chain_ids]
volumes = np.array([4935, 4529, 3466, 2111, 2624, 1738, 3110, 2919])

# volumes = np.array(hyper["num_atoms"])
# for i in range(len(file_paths_x)):
#     print(mrc_volume())
#     volumes = np.array(hyper["num_atoms"])



In [ ]:
n_points = 1000


optimal_cost = 1e10
optimal_idx = 0
points_x = []
index = []
for i in range(len(file_paths_x)):
    file_path = file_paths_x[i]
    num = int(n_points / volumes.sum() * volumes[i])
    x, y, z = sample(file_path, 0.03, num, random_seed=1)
    print(file_path, num)
    points_x.append([])
    for j in range(num):
        index.append(i)
        points_x[-1].append([x[j],y[j],z[j]])
    points_x[-1] = np.array(points_x[-1])

    # print(points_x[-1].shape)
    print("cluster %d is sampled."%(i,))


points_y = []
x, y, z = sample(file_path_y, 0.03, n_points, random_seed=1)
print(file_path_y, n_points)
for j in range(n_points):
    points_y.append([x[j],y[j],z[j]])

points_y = np.array(points_y)

print(points_y.shape)
print("points_y is sampled.")

mu, dif, costs = jgw_wrapper(points_x, points_y, epsilon=1e3, max_iter=500, ot_solver='emd', diff_thrsh=1e-2)

print("JGW finished")


x = []
y = []
z = []
for i in range(points_y.shape[0]):
    x.append(points_y[i, 0])
    y.append(points_y[i, 1])
    z.append(points_y[i, 2])

ss = 0
for k in range(len(file_paths_x)):
    x1 = []
    y1 = []
    z1 = []
    for i in range(points_x[k].shape[0]):
        x1.append(points_x[k][i, 0])
        y1.append(points_x[k][i, 1])
        z1.append(points_x[k][i, 2])

    all_coup = build_coup(x, y, z, x1, y1, z1, mu)

    Abar, Bbar, r = find_optimal_alignment(x, y, z, x1, y1, z1, all_coup)
    print("Alignment of complex %s chain %d"%(name, k))
    print("Abar: ", Abar)
    print("Bbar: ", Bbar)
    print("r: ", r.as_rotvec(degrees=True))
    print("Translational diff: ", np.linalg.norm(Abar - Bbar))
    print("Rotational diff: ", np.linalg.norm(r.as_rotvec(degrees=True)))
    alignment = {'Abar':Abar, 'Bbar':Bbar, 'r':r}

    # os.makedirs("data/output/" + name + "chains/", exist_ok=True)
    out_path = (name + "_JGW_output_%s.pdb")%(chain_ids[k])
    generate_pdb_no_mrc((base_path + name + "chains/" + name + "_chain_%s.pdb")%(chain_ids[k],), out_path, alignment)

    ss += len(x1)